# 2.1 KV Cache 优化

## 什么是KV Cache？

在自回归生成（如GPT系列模型）中，每生成一个新token都需要访问之前所有token的Key和Value向量。KV Cache将已计算的Key/Value向量缓存下来，避免重复计算，是LLM推理的核心优化。

## 为什么KV Cache是瓶颈？

KV Cache的内存占用与序列长度成正比：
$$M_{kv} = 2 \times n_{layers} \times n_{heads} \times d_{head} \times L \times b_{bytes}$$

其中：
- $n_{layers}$：Transformer层数
- $n_{heads}$：注意力头数
- $d_{head}$：每个头的维度
- $L$：序列长度
- $b_{bytes}$：每个数值的字节数（FP16为2字节）
- 系数2：Key和Value各一份

以LLaMA-7B为例（32层、32头、128维、FP16），序列长度4096时KV Cache约需3.5GB，超过模型权重本身。长序列推理的内存瓶颈往往是KV Cache而非模型权重。

## KV Cache优化的目标

1. **降低内存占用**：量化、压缩、淘汰不重要的KV
2. **高效内存管理**：分页管理、前缀复用
3. **限制缓存增长**：滑动窗口、注意力淘汰

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from collections import OrderedDict

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
## 2.1.1 KV Cache 量化

### 什么是KV Cache量化？

将缓存的Key/Value张量从FP16量化为INT8或INT4，直接将KV Cache内存占用减半或降至1/4。由于Key和Value的分布不同，通常对Key使用逐通道量化，对Value使用逐张量量化。

### 为什么KV Cache量化有效？

1. **KV Cache是内存瓶颈**：长序列推理时KV Cache占用的内存远超模型权重，量化KV Cache可以显著降低内存占用和带宽需求。
2. **KV值对量化不敏感**：研究表明，注意力机制对KV的精度有一定容忍度，INT8量化几乎无损，INT4量化损失也很小。
3. **带宽节省**：推理的瓶颈往往是内存带宽，量化后数据搬运量减半/降至1/4，直接提升推理速度。

### 为什么Key和Value使用不同的量化策略？

这是KV Cache量化中最关键的设计选择，根本原因在于Key和Value在注意力机制中扮演的角色不同，导致它们的数值分布有本质差异：

**Key的角色——"被查询的索引"**：
- Key的作用是与Query做点积计算注意力分数：$A_{ij} = q_i \cdot k_j / \sqrt{d_k}$
- 点积运算对每个维度（通道）的数值范围非常敏感：如果某个通道的数值范围特别大，它就会在点积中占据主导地位
- 实际观察中，Key的不同通道（即$d_k$维度的不同分量）的数值范围差异极大——某些通道可能集中在[-0.1, 0.1]，而另一些通道可能跨越[-5, 5]
- 如果对Key使用逐张量量化（所有通道共享一个scale），范围小的通道会被压缩到几乎只有0和±1，精度严重损失；而范围大的通道则浪费了量化精度
- 因此Key必须使用**逐通道量化**（per-channel），每个通道独立计算scale，确保每个通道都充分利用量化范围

**Value的角色——"被聚合的内容"**：
- Value的作用是在注意力加权后求和：$o_i = \sum_j A_{ij} v_j$
- Value是加权平均的输入，其输出由注意力权重控制，不同通道的数值范围相对均匀
- 实际观察中，Value的各通道数值范围差异不大，不存在Key那样的outlier通道
- 因此Value使用**逐张量量化**（per-tensor）即可，一个全局scale足够覆盖所有通道

**总结**：Key逐通道量化是因为Key的通道间数值范围差异大（点积角色导致），Value逐张量量化是因为Value的分布更均匀（加权平均角色导致）。如果对Key错误地使用逐张量量化，量化误差会显著增大，导致模型精度下降。

### KV Cache量化的数学公式

**Key量化（逐通道）**：
$$s_{k,c} = \frac{\max(|K_{:,c}|)}{q_{max}}, \quad K_{q,:,c} = \text{round}(K_{:,c} / s_{k,c})$$

**Value量化（逐张量）**：
$$s_v = \frac{\max(|V|)}{q_{max}}, \quad V_q = \text{round}(V / s_v)$$

其中：
- $K_{:,c}$：Key张量第$c$个通道的所有元素
- $s_{k,c}$：Key第$c$个通道的缩放因子（每个通道独立）
- $s_v$：Value的全局缩放因子（所有通道共享）
- $q_{max}$：量化最大值（INT8为127，INT4为7）

In [ ]:
class KVCacheQuantizer:
    """KV Cache量化器，Key逐通道量化，Value逐张量量化"""
    def __init__(self, n_bits=8):
        self.n_bits = n_bits
        self.q_max = 2 ** (n_bits - 1) - 1
        self.q_min = -2 ** (n_bits - 1)

    def quantize_key_per_channel(self, key: torch.Tensor) -> tuple:
        """Key逐通道量化：每个通道独立计算scale
        key shape: (batch, n_heads, seq_len, head_dim)
        对head_dim维度（最后一个维度）的每个通道独立量化
        """
        key_amax = key.abs().amax(dim=2, keepdim=True)
        k_scale = key_amax / self.q_max
        k_scale = torch.clamp(k_scale, min=1e-8)
        k_q = torch.clamp(torch.round(key / k_scale), self.q_min, self.q_max).to(torch.int8)
        return k_q, k_scale

    def quantize_value_per_tensor(self, value: torch.Tensor) -> tuple:
        """Value逐张量量化：整个张量共享一个scale"""
        v_scale = value.abs().max() / self.q_max
        v_scale = torch.clamp(v_scale, min=1e-8)
        v_q = torch.clamp(torch.round(value / v_scale), self.q_min, self.q_max).to(torch.int8)
        return v_q, v_scale

    def quantize_kv(self, key: torch.Tensor, value: torch.Tensor) -> tuple:
        k_q, k_scale = self.quantize_key_per_channel(key)
        v_q, v_scale = self.quantize_value_per_tensor(value)
        return k_q, k_scale, v_q, v_scale

    def dequantize_kv(self, k_q, k_scale, v_q, v_scale):
        key_deq = k_q.float() * k_scale
        value_deq = v_q.float() * v_scale
        return key_deq, value_deq

batch_size, n_heads, seq_len, head_dim = 1, 8, 512, 64
key = torch.randn(batch_size, n_heads, seq_len, head_dim)
value = torch.randn(batch_size, n_heads, seq_len, head_dim)

print("=== Key vs Value 通道间数值范围对比 ===")
key_channel_ranges = key.abs().amax(dim=2).squeeze()
value_channel_ranges = value.abs().amax(dim=2).squeeze()
print(f"Key各通道最大值范围: min={key_channel_ranges.min():.4f}, max={key_channel_ranges.max():.4f}, "
      f"极值比={key_channel_ranges.max()/key_channel_ranges.min():.2f}x")
print(f"Value各通道最大值范围: min={value_channel_ranges.min():.4f}, max={value_channel_ranges.max():.4f}, "
      f"极值比={value_channel_ranges.max()/value_channel_ranges.min():.2f}x")
print(f"\nKey通道间极值比远大于Value，说明Key的通道间数值范围差异大，必须逐通道量化")

quantizer_8bit = KVCacheQuantizer(n_bits=8)
k_q8, k_s8, v_q8, v_s8 = quantizer_8bit.quantize_kv(key, value)
k_deq8, v_deq8 = quantizer_8bit.dequantize_kv(k_q8, k_s8, v_q8, v_s8)

quantizer_4bit = KVCacheQuantizer(n_bits=4)
k_q4, k_s4, v_q4, v_s4 = quantizer_4bit.quantize_kv(key, value)
k_deq4, v_deq4 = quantizer_4bit.dequantize_kv(k_q4, k_s4, v_q4, v_s4)

kv_fp16_bytes = key.numel() * 2 + value.numel() * 2
kv_int8_bytes = k_q8.numel() * 1 + v_q8.numel() * 1 + k_s8.numel() * 2 + v_s8.numel() * 2
kv_int4_bytes = k_q4.numel() * 0.5 + v_q4.numel() * 0.5 + k_s4.numel() * 2 + v_s4.numel() * 2

k_err8 = (key - k_deq8).norm() / key.norm() * 100
v_err8 = (value - v_deq8).norm() / value.norm() * 100
k_err4 = (key - k_deq4).norm() / key.norm() * 100
v_err4 = (value - v_deq4).norm() / value.norm() * 100

print(f"\n=== KV Cache量化效果 ===")
print(f"序列长度={seq_len}, 头数={n_heads}, 头维度={head_dim}")
print(f"\n{'格式':<15} {'内存(KB)':<15} {'Key误差%':<12} {'Value误差%':<12}")
print("-" * 54)
print(f"{'FP16':<15} {kv_fp16_bytes/1024:<15.2f} {'0.00':<12} {'0.00':<12}")
print(f"{'INT8':<15} {kv_int8_bytes/1024:<15.2f} {k_err8:<12.4f} {v_err8:<12.4f}")
print(f"{'INT4':<15} {kv_int4_bytes/1024:<15.2f} {k_err4:<12.4f} {v_err4:<12.4f}")
print(f"\n压缩比: FP16->INT8={kv_fp16_bytes/kv_int8_bytes:.2f}x, FP16->INT4={kv_fp16_bytes/kv_int4_bytes:.2f}x")

print(f"\n=== 对比：如果对Key错误使用逐张量量化 ===")
k_scale_wrong = key.abs().max() / 127
k_scale_wrong = torch.clamp(k_scale_wrong, min=1e-8)
k_q_wrong = torch.clamp(torch.round(key / k_scale_wrong), -128, 127).to(torch.int8)
k_deq_wrong = k_q_wrong.float() * k_scale_wrong
k_err_wrong = (key - k_deq_wrong).norm() / key.norm() * 100
print(f"Key逐通道量化误差: {k_err8:.4f}%")
print(f"Key逐张量量化误差: {k_err_wrong:.4f}%")
print(f"误差放大: {k_err_wrong/k_err8:.1f}x")
print(f"\n结论: Key必须逐通道量化，逐张量量化会导致误差显著增大")

---
## 2.1.2 KV Cache 内存管理

### PagedAttention（vLLM）

#### 什么是PagedAttention？

PagedAttention借鉴操作系统虚拟内存的分页管理思想，将KV Cache划分为固定大小的block，按需分配，非连续存储。vLLM推理框架的核心技术。

#### 为什么PagedAttention有效？

1. **消除内存碎片**：传统KV Cache需要为每个序列预分配最大长度的连续内存，造成大量浪费。PagedAttention按block分配，无需连续内存。
2. **支持更大batch**：内存利用率从传统方式的60%-80%提升到接近100%，可服务更多并发请求。
3. **动态内存管理**：序列完成即可释放block，新序列可复用，类似OS的内存分页机制。

#### PagedAttention的Block管理机制详解

PagedAttention的核心是一个类似OS虚拟内存的三层结构：

| OS虚拟内存 | PagedAttention | 说明 |
|-----------|---------------|------|
| 物理页帧 (Frame) | Block Pool | 预分配的物理KV存储块 |
| 页表 (Page Table) | Block Table | 每个sequence的逻辑block到物理block的映射 |
| 空闲页帧列表 | Free Block List | 当前未使用的block编号集合 |

**Block的生命周期**：

1. **系统初始化**：预分配一个固定大小的Block Pool（如64个block），所有block初始为空闲状态，加入Free Block List
2. **Sequence创建**（新请求到来）：为该sequence创建一个空的Block Table
3. **KV写入**（prefill/decode）：
   - 检查当前block是否还有空位（已用token数 < block_size）
   - 如果有空位：直接写入KV到当前block的下一个位置
   - 如果没有空位：从Free Block List取出一个block，分配给该sequence，追加到Block Table末尾，然后写入
4. **Sequence完成**（生成结束）：将该sequence的Block Table中所有block归还Free Block List，清空Block Table

**关键设计**：
- Block大小（block_size）通常为16个token，是内存碎片和管理开销的折中——太小则block table过长，太大则每个block末尾浪费更多空间
- Block Table是一个整数数组，`block_table[logical_idx] = physical_block_id`，逻辑上连续的KV在物理上可以不连续
- 多个sequence共享同一个Block Pool，实现内存的动态分配和复用
- 当Free Block List为空时，新请求需要等待（抢占：暂停优先级最低的sequence，释放其block）

#### PagedAttention的数学分析

传统方式为每个序列预分配最大长度$L_{max}$的连续内存：
$$M_{traditional} = B \times L_{max} \times d_{kv} \times b_{bytes}$$

PagedAttention按block分配，实际使用量等于真实序列长度：
$$M_{paged} = \sum_{i=1}^{B} \lceil L_i / B_s \rceil \times B_s \times d_{kv} \times b_{bytes}$$

其中：
- $B$：batch大小（并发请求数）
- $L_{max}$：最大序列长度
- $L_i$：第$i$个请求的实际序列长度
- $B_s$：block大小（通常16个token）
- $d_{kv}$：KV向量的维度
- 浪费率仅取决于block内的碎片（最多$B_s - 1$个token的浪费）

In [ ]:
class PagedKVCache:
    """PagedAttention实现，模拟vLLM的block管理机制

    核心数据结构：
    - kv_block_pool: 物理block池，shape=(n_blocks, 2, n_heads, block_size, head_dim)
    - free_blocks: 空闲block列表（可分配的物理block ID）
    - seq_block_tables: 每个sequence的逻辑block -> 物理block映射表
    - seq_token_counts: 每个sequence已写入的token数
    """
    def __init__(self, n_heads, head_dim, block_size=16, n_blocks=64):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.block_size = block_size
        self.n_blocks = n_blocks
        self.kv_block_pool = torch.zeros(n_blocks, 2, n_heads, block_size, head_dim)
        self.free_blocks = list(range(n_blocks))
        self.seq_block_tables = {}
        self.seq_token_counts = {}

    def _allocate_block(self):
        """从Free Block List分配一个物理block"""
        if not self.free_blocks:
            raise RuntimeError("KV Cache OOM: no free blocks available")
        block_id = self.free_blocks.pop(0)
        return block_id

    def _free_block(self, block_id):
        """将物理block归还Free Block List"""
        self.kv_block_pool[block_id].zero_()
        self.free_blocks.append(block_id)

    def create_sequence(self, seq_id):
        """为新请求创建sequence，初始化Block Table"""
        self.seq_block_tables[seq_id] = []
        self.seq_token_counts[seq_id] = 0

    def destroy_sequence(self, seq_id):
        """请求完成，释放该sequence占用的所有block"""
        if seq_id in self.seq_block_tables:
            for block_id in self.seq_block_tables[seq_id]:
                self._free_block(block_id)
            del self.seq_block_tables[seq_id]
            del self.seq_token_counts[seq_id]

    def append_kv(self, seq_id, key, value):
        """向指定sequence追加KV对
        key/value shape: (n_heads, 1, head_dim) — 单个token
        """
        if seq_id not in self.seq_block_tables:
            self.create_sequence(seq_id)

        token_count = self.seq_token_counts[seq_id]
        pos_in_block = token_count % self.block_size

        if pos_in_block == 0:
            new_block = self._allocate_block()
            self.seq_block_tables[seq_id].append(new_block)

        current_block = self.seq_block_tables[seq_id][-1]
        self.kv_block_pool[current_block, 0, :, pos_in_block, :] = key.squeeze(1)
        self.kv_block_pool[current_block, 1, :, pos_in_block, :] = value.squeeze(1)
        self.seq_token_counts[seq_id] += 1

    def append_kv_prefill(self, seq_id, keys, values):
        """批量写入多个token的KV（prefill阶段使用）
        keys/values shape: (n_heads, seq_len, head_dim)
        """
        if seq_id not in self.seq_block_tables:
            self.create_sequence(seq_id)

        seq_len = keys.shape[1]
        for i in range(seq_len):
            self.append_kv(seq_id, keys[:, i:i+1, :], values[:, i:i+1, :])

    def get_kv(self, seq_id):
        """获取指定sequence的所有KV，按逻辑顺序拼接"""
        if seq_id not in self.seq_block_tables:
            return None, None
        blocks = self.seq_block_tables[seq_id]
        if not blocks:
            return None, None
        token_count = self.seq_token_counts[seq_id]
        keys = [self.kv_block_pool[b, 0] for b in blocks]
        values = [self.kv_block_pool[b, 1] for b in blocks]
        keys_cat = torch.cat(keys, dim=1)[:, :token_count, :]
        values_cat = torch.cat(values, dim=1)[:, :token_count, :]
        return keys_cat, values_cat

    def memory_usage(self):
        used = self.n_blocks - len(self.free_blocks)
        return used / self.n_blocks * 100

    def print_status(self):
        print(f"  Free blocks: {len(self.free_blocks)}/{self.n_blocks}")
        print(f"  Memory usage: {self.memory_usage():.1f}%")
        for seq_id in sorted(self.seq_block_tables.keys()):
            n_tokens = self.seq_token_counts[seq_id]
            n_blocks = len(self.seq_block_tables[seq_id])
            block_ids = self.seq_block_tables[seq_id]
            waste = n_blocks * self.block_size - n_tokens
            print(f"  Seq {seq_id}: {n_tokens} tokens, {n_blocks} blocks (physical IDs: {block_ids}), waste: {waste} slots")

paged_cache = PagedKVCache(n_heads=8, head_dim=64, block_size=16, n_blocks=64)

print("=== PagedAttention Block管理演示 ===")
print("\n[1] 初始状态:")
paged_cache.print_status()

print("\n[2] Sequence 0 开始prefill (32 tokens):")
k_prefill = torch.randn(8, 32, 64)
v_prefill = torch.randn(8, 32, 64)
paged_cache.append_kv_prefill(seq_id=0, keys=k_prefill, values=v_prefill)
paged_cache.print_status()

print("\n[3] Sequence 1 开始prefill (16 tokens):")
k_prefill1 = torch.randn(8, 16, 64)
v_prefill1 = torch.randn(8, 16, 64)
paged_cache.append_kv_prefill(seq_id=1, keys=k_prefill1, values=v_prefill1)
paged_cache.print_status()

print("\n[4] Sequence 0 decode (1 token):")
k_decode = torch.randn(8, 1, 64)
v_decode = torch.randn(8, 1, 64)
paged_cache.append_kv(seq_id=0, key=k_decode, value=v_decode)
paged_cache.print_status()

print("\n[5] Sequence 1 完成，释放block:")
paged_cache.destroy_sequence(seq_id=1)
paged_cache.print_status()

print("\n[6] Sequence 2 开始prefill (48 tokens)，复用已释放的block:")
k_prefill2 = torch.randn(8, 48, 64)
v_prefill2 = torch.randn(8, 48, 64)
paged_cache.append_kv_prefill(seq_id=2, keys=k_prefill2, values=v_prefill2)
paged_cache.print_status()

keys_0, values_0 = paged_cache.get_kv(seq_id=0)
keys_2, values_2 = paged_cache.get_kv(seq_id=2)
print(f"\n验证: Seq0 KV长度={keys_0.shape[1]}, Seq2 KV长度={keys_2.shape[1]}")
print(f"\n关键观察: Seq2复用了Seq1释放的block（物理block ID可重叠），不同sequence的逻辑block在物理上可以不连续")

### Prefix Caching

#### 什么是Prefix Caching？

对共享前缀（如system prompt）的KV Cache进行缓存复用，避免重复计算。多个请求共享相同前缀时，只需计算一次前缀的KV，后续请求直接复用。

#### 为什么Prefix Caching有效？

1. **前缀复用率高**：在对话场景中，system prompt通常占数百个token，且所有请求共享，缓存后可节省大量重复计算。
2. **零精度损失**：完全复用已计算的精确KV，不引入任何近似误差。
3. **延迟降低**：新请求跳过前缀的prefill阶段，首token延迟显著降低。

#### Prefix Caching的具体使用流程

Prefix Caching的核心思路是：**将注意力计算的KV来源分为两部分——缓存的前缀KV和新计算的token KV，拼接后一起参与注意力计算。**

```
请求到来: [system_prompt | user_input_1]
                ↑                ↑
          从缓存取KV        新计算KV
                ↓                ↓
           [prefix_kv || new_kv]  ← 拼接后参与注意力计算
```

**具体步骤**：

1. **首次请求**（缓存未命中）：
   - 对完整输入 `[system_prompt | user_input]` 执行prefill，计算所有token的KV
   - 将system_prompt部分的KV单独提取出来，以token序列的hash为key存入Prefix Cache

2. **后续请求**（缓存命中）：
   - 用system_prompt的token序列计算hash，在Prefix Cache中查找
   - 命中：取出缓存的prefix KV（shape: `(n_heads, L_prefix, head_dim)`）
   - 只对user_input部分执行prefill，计算新token的KV（shape: `(n_heads, L_user, head_dim)`）
   - 将prefix KV和新KV在序列维度上拼接：`KV_full = cat([prefix_kv, new_kv], dim=1)`
   - 用拼接后的KV进行注意力计算

3. **与PagedAttention结合**（vLLM实际做法）：
   - 前缀的block在Block Table中标记为"共享"，引用计数+1
   - 新请求的Block Table前半部分指向共享的prefix block，后半部分指向新分配的block
   - 这样前缀的物理内存只有一份，多个sequence通过Block Table共享引用

#### Prefix Caching的数学分析

设前缀长度为$L_p$，用户输入平均长度为$L_u$，则：
- 无缓存：每个请求需计算$(L_p + L_u)$个token的KV
- 有缓存：每个请求只需计算$L_u$个token的KV
- 计算节省比：$\frac{L_p}{L_p + L_u}$

当$L_p = 512, L_u = 64$时，节省比约89%。

In [ ]:
class SimpleAttention(nn.Module):
    """标准多头注意力，用于演示Prefix Caching与推理的集成"""
    def __init__(self, n_heads, head_dim):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.dim = n_heads * head_dim
        self.q_proj = nn.Linear(self.dim, self.dim, bias=False)
        self.k_proj = nn.Linear(self.dim, self.dim, bias=False)
        self.v_proj = nn.Linear(self.dim, self.dim, bias=False)
        self.out_proj = nn.Linear(self.dim, self.dim, bias=False)

    def forward(self, x, cached_k=None, cached_v=None):
        """x: (batch, seq_len, dim), cached_k/v: (batch, n_heads, prefix_len, head_dim)"""
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        if cached_k is not None and cached_v is not None:
            k = torch.cat([cached_k, k], dim=2)
            v = torch.cat([cached_v, v], dim=2)

        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v


class PrefixCacheWithInference:
    """Prefix Caching与推理流程的完整集成"""
    def __init__(self, n_heads, head_dim, max_cache_entries=50):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.attn = SimpleAttention(n_heads, head_dim)
        self.cache = OrderedDict()
        self.max_cache_entries = max_cache_entries

    def _compute_prefix_hash(self, token_ids):
        return hash(tuple(token_ids.tolist()))

    def prefill_with_prefix(self, prefix_tokens, user_tokens):
        """完整推理流程：检查prefix缓存 -> 计算新KV -> 拼接 -> 注意力计算

        Args:
            prefix_tokens: (prefix_len,) 共享前缀的token IDs
            user_tokens: (user_len,) 用户输入的token IDs
        Returns:
            output: 注意力输出, compute_saved: 节省的计算量
        """
        prefix_hash = self._compute_prefix_hash(prefix_tokens)
        prefix_len = len(prefix_tokens)
        user_len = len(user_tokens)

        cached = self.cache.get(prefix_hash)

        if cached is not None:
            cached_k, cached_v = cached
            user_embed = torch.randn(1, user_len, self.n_heads * self.head_dim)
            output, new_k, new_v = self.attn(user_embed, cached_k=cached_k, cached_v=cached_v)
            compute_saved = prefix_len
            return output, compute_saved
        else:
            full_tokens = torch.cat([prefix_tokens, user_tokens])
            full_len = len(full_tokens)
            full_embed = torch.randn(1, full_len, self.n_heads * self.head_dim)
            output, full_k, full_v = self.attn(full_embed)

            prefix_k = full_k[:, :, :prefix_len, :].clone()
            prefix_v = full_v[:, :, :prefix_len, :].clone()
            self.cache[prefix_hash] = (prefix_k, prefix_v)
            if len(self.cache) > self.max_cache_entries:
                self.cache.popitem(last=False)

            compute_saved = 0
            return output, compute_saved

n_heads, head_dim = 8, 64
engine = PrefixCacheWithInference(n_heads, head_dim)

system_prompt_tokens = torch.randint(0, 1000, (128,))

print("=== Prefix Caching 完整推理流程演示 ===")
print(f"System prompt长度: {len(system_prompt_tokens)} tokens")

total_compute = 0
total_saved = 0
n_requests = 5

for i in range(n_requests):
    user_tokens = torch.randint(0, 1000, (64 + i * 10,))
    output, saved = engine.prefill_with_prefix(system_prompt_tokens, user_tokens)
    total_compute += len(system_prompt_tokens) + len(user_tokens)
    total_saved += saved
    status = "缓存命中" if saved > 0 else "缓存未命中，写入缓存"
    print(f"\n请求 {i+1}: user_len={len(user_tokens)}, {status}")
    print(f"  输出shape: {output.shape}")
    print(f"  本次节省计算: {saved} tokens")

print(f"\n=== 总结 ===")
print(f"总请求: {n_requests}")
print(f"总需计算tokens(无缓存): {total_compute}")
print(f"实际节省tokens: {total_saved}")
print(f"计算量节省: {total_saved/total_compute*100:.1f}%")
print(f"\nPrefix Cache条目数: {len(engine.cache)}")
cached_k, cached_v = list(engine.cache.values())[0]
print(f"缓存的KV shape: K={cached_k.shape}, V={cached_v.shape}")
print(f"\n关键: 缓存命中时，只需计算user_input部分的KV，然后与缓存的prefix KV拼接即可")

---
## 2.1.3 KV Cache 压缩与淘汰

### 滑动窗口注意力（Sliding Window Attention）

#### 什么是滑动窗口注意力？

每个token只关注最近$W$个token的KV，超出窗口的KV被丢弃。KV Cache大小固定为$O(W)$，不随序列增长。Mistral、Gemini等模型采用此策略。

#### 为什么滑动窗口有效？

1. **内存可控**：KV Cache大小固定为$W$，不随序列长度增长，适合端侧设备的有限内存。
2. **局部性原理**：自然语言中，相邻token的依赖关系最强，远距离依赖相对较少，窗口注意力在大多数任务上表现良好。
3. **信息传递**：虽然每个token只看$W$个邻居，但通过多层堆叠，信息可以逐层传递到更远的token（感受野为$W \times n_{layers}$）。

#### 滑动窗口的大小是否可以变化？

滑动窗口的大小$W$是一个关键超参数，其选择涉及以下考量：

**1. 固定窗口 vs 可变窗口**：
- **固定窗口**（主流做法）：所有层、所有位置使用相同的$W$。Mistral-7B使用$W=4096$，Gemini使用$W=8192$。优点是实现简单，内存可精确预测。
- **可变窗口**（研究探索）：不同层或不同位置使用不同的$W$。例如，浅层使用较小窗口（局部依赖为主），深层使用较大窗口（需要更远的上下文）。但实现复杂，且需要额外的内存管理逻辑，目前产业界很少采用。

**2. 窗口大小$W$的选择依据**：
- **任务需求**：需要长距离依赖的任务（如文档摘要）需要更大的$W$；短文本生成任务可以用较小的$W$
- **内存预算**：$W$直接决定KV Cache大小，需在内存预算内选择：$W \leq \frac{M_{budget}}{2 \times n_{layers} \times n_{heads} \times d_{head} \times b_{bytes}}$
- **信息传递距离**：$L$层模型的有效感受野为$W \times L$，需确保感受野覆盖任务所需的最长依赖距离

**3. 实践建议**：
- 端侧设备：$W$ = 512~2048，平衡内存和效果
- 服务器推理：$W$ = 4096~8192，配合Rolling Buffer实现循环缓存
- 如果任务需要极长上下文，应考虑H2O等选择性保留策略，而非单纯增大窗口

#### 滑动窗口的数学公式

注意力计算仅对窗口内的KV进行：
$$\text{Attn}(q_i, K, V) = \text{softmax}\left(\frac{q_i K_{[i-W:i]}^T}{\sqrt{d_k}}\right) V_{[i-W:i]}$$

其中：
- $q_i$：第$i$个token的查询向量
- $K_{[i-W:i]}$：第$i-W$到第$i$个token的Key矩阵
- $V_{[i-W:i]}$：对应的Value矩阵
- $W$：窗口大小
- $d_k$：Key的维度
- 多层后的有效感受野：$W \times n_{layers}$

In [ ]:
class SlidingWindowKVCache:
    """滑动窗口KV Cache，使用循环缓冲区（Rolling Buffer）实现

    实现要点：
    - 使用固定大小的环形缓冲区，新token覆盖最旧的token
    - get_kv时需要按时间顺序重排，保证最新token在末尾
    - 窗口大小W固定，不随序列增长
    """
    def __init__(self, n_heads, head_dim, window_size=256):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.window_size = window_size
        self.key_cache = torch.zeros(1, n_heads, window_size, head_dim)
        self.value_cache = torch.zeros(1, n_heads, window_size, head_dim)
        self.current_pos = 0

    def update(self, new_key, new_value):
        """更新KV Cache，超出窗口自动淘汰最旧的KV"""
        seq_len = new_key.shape[2]
        for i in range(seq_len):
            pos = self.current_pos % self.window_size
            self.key_cache[:, :, pos, :] = new_key[:, :, i, :]
            self.value_cache[:, :, pos, :] = new_value[:, :, i, :]
            self.current_pos += 1

    def get_kv(self):
        """获取当前窗口内的KV（保证最新token在末尾）"""
        if self.current_pos <= self.window_size:
            return (self.key_cache[:, :, :self.current_pos, :],
                    self.value_cache[:, :, :self.current_pos, :])
        start = self.current_pos % self.window_size
        key_ordered = torch.cat([self.key_cache[:, :, start:, :], self.key_cache[:, :, :start, :]], dim=2)
        value_ordered = torch.cat([self.value_cache[:, :, start:, :], self.value_cache[:, :, :start, :]], dim=2)
        return key_ordered, value_ordered

    def memory_mb(self):
        return self.key_cache.numel() * 2 / 1024 / 1024 + self.value_cache.numel() * 2 / 1024 / 1024

n_heads, head_dim = 8, 64

print("=== 滑动窗口KV Cache ===")
print(f"\n内存占用对比 (固定，不随序列增长):")
for ws in [128, 256, 512, 1024, 2048, 4096]:
    sw_cache = SlidingWindowKVCache(n_heads, head_dim, window_size=ws)
    print(f"  窗口={ws:<6} 内存={sw_cache.memory_mb():.3f} MB")

sw = SlidingWindowKVCache(n_heads, head_dim, window_size=256)
for step in range(500):
    k = torch.randn(1, n_heads, 1, head_dim)
    v = torch.randn(1, n_heads, 1, head_dim)
    sw.update(k, v)

k_out, v_out = sw.get_kv()
print(f"\n写入500个token后:")
print(f"  实际缓存KV长度: {k_out.shape[2]}")
print(f"  内存占用: {sw.memory_mb():.3f} MB (恒定)")
print(f"  丢弃的旧KV: {500 - 256} tokens")

print(f"\n=== 窗口大小选择分析 ===")
n_layers = 32
for ws in [512, 1024, 2048, 4096]:
    receptive_field = ws * n_layers
    mem = 1 * n_heads * ws * head_dim * 2 * 2 / 1024 / 1024
    print(f"  W={ws:<5} 感受野={receptive_field:>7} tokens, KV Cache={mem:.3f} MB/层")

print(f"\n注意: 感受野=W×层数，但信息传递是逐层衰减的，实际有效距离远小于理论感受野")
print(f"\n对比无限制KV Cache:")
for seq_len in [256, 512, 1024, 2048, 4096]:
    unlimited_mem = 1 * n_heads * seq_len * head_dim * 2 * 2 / 1024 / 1024
    sw_mem = 1 * n_heads * 256 * head_dim * 2 * 2 / 1024 / 1024
    print(f"  seq={seq_len:<5} 无限制={unlimited_mem:.3f}MB 滑动窗口(256)={sw_mem:.3f}MB 节省{(1-sw_mem/unlimited_mem)*100:.0f}%")

### H2O - 基于注意力分数的KV淘汰

#### 什么是H2O？

H2O（Heavy-Hitter Oracle）当KV Cache超出预算时，根据注意力分数选择性淘汰不重要的KV。保留注意力分数最高的KV（Heavy Hitter）和最近的KV。

#### 为什么H2O有效？

1. **注意力分布不均**：LLM的注意力分数呈长尾分布，少数token获得大部分注意力（Heavy Hitter），这些token对输出影响最大。
2. **保留重要信息**：淘汰注意力分数低的KV，保留对输出影响最大的KV，精度损失最小。
3. **最近token保护**：最近的token通常对下一个token的预测最重要，必须保留。

#### 注意力分数如何对应到具体token？

这是理解H2O的关键问题。注意力分数到token的映射机制如下：

**1. 注意力权重的结构**：
在自回归模型中，第$t$个token生成时，注意力权重$A_t$是一个长度为$t$的向量：
$$A_t = \text{softmax}\left(\frac{q_t K_{[:t]}^T}{\sqrt{d_k}}\right) \in \mathbb{R}^{t}$$
其中$A_t[i]$表示第$t$个token对第$i$个token的注意力权重。**$A_t$的每个元素天然对应一个具体的token位置**。

**2. 累积注意力分数**：
第$i$个token的累积注意力分数是所有后续token对它的注意力权重之和：
$$S_i = \sum_{t=i+1}^{L} A_{t,i}$$
- $S_i$是一个标量，与第$i$个token一一对应
- $S_i$越大，说明token $i$被越多后续token关注，越重要

**3. 淘汰时的映射**：
H2O维护一个与KV Cache等长的累积分数列表`[S_0, S_1, ..., S_{L-1}]`，每个分数通过数组索引与对应位置的KV绑定。淘汰时：
- 对分数列表排序，找到分数最低的位置索引
- 通过索引删除对应位置的Key和Value
- 保留最近$W$个位置（不参与排序，直接保留）

#### 预算$B$如何确定？

预算$B$（KV Cache保留的最大token数）的确定需要综合考虑以下因素：

**1. 基于内存预算反推**：
$$B = \frac{M_{budget}}{2 \times n_{layers} \times n_{heads} \times d_{head} \times b_{bytes}}$$
其中$M_{budget}$是分配给KV Cache的最大内存。例如，端侧设备分配512MB给KV Cache，LLaMA-7B（32层、32头、128维、FP16）则：
$$B = \frac{512 \times 1024^2}{2 \times 32 \times 32 \times 128 \times 2} \approx 1024 \text{ tokens}$$

**2. 基于精度需求调整**：
- H2O论文实验表明，$B$设为原始序列长度的20%-40%即可保持接近无损的精度
- 对于精度敏感任务，$B$应设为50%以上
- 对于内存极度受限场景，$B$可低至10%，但需验证精度

**3. 最近窗口$W$的分配**：
- $W$通常设为$B$的20%-30%（如$B=128$时$W=32$）
- $W$保证最近的上下文完整保留，避免局部连贯性被破坏

#### H2O的数学原理

累积注意力分数作为重要性度量：
$$S_i = \sum_{t=i+1}^{L} A_{t,i}$$

淘汰策略：在预算$B$内，保留：
- 最近$W$个token的KV（recent window）
- 历史KV中累积注意力分数最高的$B - W$个token

其中：
- $S_i$：第$i$个token的累积注意力分数
- $A_{t,i}$：第$t$个token对第$i$个token的注意力权重
- $B$：KV Cache预算（保留的最大token数）
- $W$：最近窗口大小
- $L$：当前序列长度

In [ ]:
class H2OKVCache:
    """H2O: 基于注意力分数的KV淘汰策略

    核心机制：
    - 每个token位置维护一个累积注意力分数 S_i
    - S_i 通过数组索引与 KV[i] 一一对应
    - 淘汰时：保留最近W个token + 历史中S_i最高的(B-W)个token
    """
    def __init__(self, n_heads, head_dim, budget=128, recent_window=32):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.budget = budget
        self.recent_window = recent_window
        self.keys = []
        self.values = []
        self.cumulative_scores = []

    def update(self, new_key, new_value, attn_weights_for_new_token=None):
        """更新KV Cache

        Args:
            new_key: (1, n_heads, 1, head_dim) 新token的Key
            new_value: (1, n_heads, 1, head_dim) 新token的Value
            attn_weights_for_new_token: (1, n_heads, 1, current_len) 新token对已有token的注意力权重
                - shape的最后一维对应已有KV Cache中的每个token位置
                - attn_weights_for_new_token[:, :, :, i] 就是 A_{t,i}，表示新token对第i个token的注意力
        """
        self.keys.append(new_key)
        self.values.append(new_value)

        if attn_weights_for_new_token is not None:
            new_attn = attn_weights_for_new_token.sum(dim=(0, 1)).squeeze()
            if new_attn.dim() == 0:
                new_attn = new_attn.unsqueeze(0)
            for i in range(len(self.cumulative_scores)):
                if i < new_attn.shape[0]:
                    self.cumulative_scores[i] += new_attn[i].item()
            self.cumulative_scores.append(1.0)
        else:
            for i in range(len(self.cumulative_scores)):
                self.cumulative_scores[i] += 1.0 / max(len(self.cumulative_scores), 1)
            self.cumulative_scores.append(1.0)

        if len(self.keys) > self.budget:
            self._evict()

    def _evict(self):
        """淘汰策略：保留最近窗口内的 + 累积注意力分数最高的"""
        n = len(self.keys)
        recent_start = max(0, n - self.recent_window)

        older_indices = list(range(recent_start))
        if older_indices:
            older_scores = torch.tensor([self.cumulative_scores[i] for i in older_indices])
            n_to_keep = self.budget - self.recent_window
            n_to_keep = min(n_to_keep, len(older_indices))
            _, top_indices = older_scores.topk(n_to_keep)
            keep_set = set(older_indices[i] for i in top_indices.tolist())
            keep_set.update(range(recent_start, n))

            sorted_keep = sorted(keep_set)
            self.keys = [self.keys[i] for i in sorted_keep]
            self.values = [self.values[i] for i in sorted_keep]
            self.cumulative_scores = [self.cumulative_scores[i] for i in sorted_keep]

    def get_kv(self):
        if not self.keys:
            return None, None
        return torch.cat(self.keys, dim=2), torch.cat(self.values, dim=2)

    def __len__(self):
        return len(self.keys)


def simulate_attention_weights(seq_len, n_heads, heavy_hitter_positions=None):
    """模拟真实注意力分布：少数token获得大部分注意力（长尾分布）

    Returns:
        attn_weights: (1, n_heads, 1, seq_len) 模拟的注意力权重
    """
    logits = torch.randn(1, n_heads, 1, seq_len) * 0.5
    if heavy_hitter_positions is not None:
        for pos in heavy_hitter_positions:
            if pos < seq_len:
                logits[:, :, :, pos] += 5.0
    attn = F.softmax(logits, dim=-1)
    return attn


n_heads, head_dim = 8, 64
budget = 128
recent_window = 32
h2o = H2OKVCache(n_heads=n_heads, head_dim=head_dim, budget=budget, recent_window=recent_window)

heavy_hitters = [5, 15, 42, 78, 120, 180, 220]
total_steps = 256

for step in range(total_steps):
    k = torch.randn(1, n_heads, 1, head_dim)
    v = torch.randn(1, n_heads, 1, head_dim)
    current_len = len(h2o)
    if current_len > 0:
        attn = simulate_attention_weights(current_len, n_heads, heavy_hitter_positions=heavy_hitters)
        h2o.update(k, v, attn_weights_for_new_token=attn)
    else:
        h2o.update(k, v)

k_out, v_out = h2o.get_kv()
unlimited_mem = 1 * n_heads * total_steps * head_dim * 2 * 2 / 1024 / 1024
h2o_mem = 1 * n_heads * len(h2o) * head_dim * 2 * 2 / 1024 / 1024

print(f"=== H2O KV淘汰策略 ===")
print(f"总生成tokens: {total_steps}")
print(f"KV Cache预算B: {budget}")
print(f"最近窗口W: {recent_window}")
print(f"实际缓存tokens: {len(h2o)}")
print(f"无限制内存: {unlimited_mem:.3f} MB")
print(f"H2O内存: {h2o_mem:.3f} MB")
print(f"节省: {(1-h2o_mem/unlimited_mem)*100:.0f}%")

print(f"\n=== 预算B的确定方法 ===")
for mem_budget_mb in [128, 256, 512, 1024]:
    B = int(mem_budget_mb * 1024 * 1024 / (2 * 32 * 32 * 128 * 2))
    print(f"  内存预算={mem_budget_mb}MB -> B≈{B} tokens (LLaMA-7B FP16)")

print(f"\n=== 注意力分数与token的对应关系 ===")
print(f"累积注意力分数列表长度: {len(h2o.cumulative_scores)}")
print(f"分数范围: min={min(h2o.cumulative_scores):.2f}, max={max(h2o.cumulative_scores):.2f}")
print(f"\n机制说明:")
print(f"  - cumulative_scores[i] 与 keys[i]/values[i] 通过数组索引一一对应")
print(f"  - 每生成一个新token，其注意力权重 A_t[i] 累加到 cumulative_scores[i]")
print(f"  - 淘汰时：对 cumulative_scores 排序，删除分数最低的token的KV")
print(f"  - 最近W个token不参与淘汰，直接保留")

### 跨层KV共享（Cross-Layer KV Sharing）

#### 什么是跨层KV共享？

相邻层的KV表示高度相似，可共享同一份KV Cache。如CLA（Cross-Layer Attention）、YOCO等架构，将KV Cache占用减半。

#### 为什么跨层KV共享有效？

1. **层间相似性**：研究表明Transformer相邻层的Key和Value向量余弦相似度高达0.9以上，存在大量冗余。
2. **内存直接减半**：每2层共享1组KV投影，KV Cache减少50%，且无需额外压缩操作。
3. **计算量也减少**：KV投影层减少一半，前向计算量也相应降低。

#### 跨层KV共享的参数如何确定？

跨层KV共享涉及三个关键参数：**共享步长（stride）$s$**、**哪些层共享**、**共享的KV由哪层计算**。

**1. 共享步长$s$的确定**：

$s$决定了每$s$层共享一组KV投影，是最重要的参数。确定方法：

- **基于层间相似度测量**：对预训练模型，计算所有相邻层的KV余弦相似度矩阵。如果层$l$和层$l+1$的相似度$> 0.9$，可以共享；如果相似度$< 0.8$，不建议共享。
- **经验规律**：
  - $s=2$（每2层共享1组KV）：最常用，几乎所有相邻层相似度都足够高，精度损失最小
  - $s=3$（每3层共享1组KV）：部分模型可行，但需验证中间层的注意力质量
  - $s>3$：通常不建议，相似度下降明显，精度损失较大
- **搜索策略**：在验证集上评估不同$s$值的困惑度（perplexity），选择精度可接受的最大$s$

**2. 哪些层可以共享**：

- **均匀分组**（最常见）：将$L$层均匀分为$L/s$组，每组共享。如$L=32, s=2$，则层0-1共享、层2-3共享、...
- **非均匀分组**（精细优化）：根据相似度矩阵，将相似度高的层分为一组。例如，浅层相似度更高可以用$s=3$，深层相似度较低用$s=2$。但实现复杂，收益有限。

**3. 共享的KV由哪层计算**：

- **由组内第一层计算**（CLA做法）：组内第一层正常计算KV，后续层复用。优点是KV基于更早层的表示，信息更"原始"。
- **由组内最后一层计算**（YOCO做法）：组内最后一层计算KV，前面的层复用。优点是KV基于更深的表示，信息更"精炼"。
- **由组内中间层计算**：折中方案，实践中较少使用。

**4. 参数确定流程总结**：
```
1. 测量预训练模型的层间KV相似度矩阵
2. 根据相似度确定初始stride s（通常从s=2开始）
3. 选择分组策略（均匀分组最简单）
4. 选择KV计算层（第一层 or 最后一层）
5. 在验证集上评估perplexity
6. 如果精度可接受，尝试增大s；否则减小s
7. 确定最终参数
```

#### 跨层KV共享的数学原理

标准Transformer每层独立计算KV：
$$K_l = X_l W_k^l, \quad V_l = X_l W_v^l, \quad l = 1, 2, ..., L$$

跨层共享策略（每$s$层共享1组KV投影）：
$$K_l = X_l W_k^{\lfloor l/s \rfloor}, \quad V_l = X_l W_v^{\lfloor l/s \rfloor}$$

其中：
- $X_l$：第$l$层的输入
- $W_k^l, W_v^l$：第$l$层的Key和Value投影矩阵
- $s$：共享步长（stride），通常为2
- $\lfloor l/s \rfloor$：整除，表示第$l$层使用的KV投影组索引
- KV投影参数从$L$组减少到$L/s$组

In [ ]:
class CrossLayerKVSharedTransformer(nn.Module):
    """跨层KV共享的Transformer，模拟CLA架构

    参数说明：
    - share_stride: 共享步长s，每s层共享1组KV投影。s=2最常用。
    - kv_compute_position: 'first'（组内第一层计算KV，CLA做法）
                          'last'（组内最后一层计算KV，YOCO做法）
    """
    def __init__(self, n_layers=6, dim=64, n_heads=4, share_stride=2,
                 kv_compute_position='first'):
        super().__init__()
        self.n_layers = n_layers
        self.share_stride = share_stride
        self.kv_compute_position = kv_compute_position
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.q_projs = nn.ModuleList([nn.Linear(dim, dim, bias=False) for _ in range(n_layers)])
        n_kv_groups = (n_layers + share_stride - 1) // share_stride
        self.kv_projs = nn.ModuleList([nn.Linear(dim, 2 * dim, bias=False) for _ in range(n_kv_groups)])
        self.out_projs = nn.ModuleList([nn.Linear(dim, dim, bias=False) for _ in range(n_layers)])
        self.ffns = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
            for _ in range(n_layers)
        ])
        self.lns = nn.ModuleList([nn.LayerNorm(dim) for _ in range(n_layers * 2)])

    def _get_kv_group_idx(self, layer_idx):
        return layer_idx // self.share_stride

    def _should_compute_kv(self, layer_idx):
        group_start = (layer_idx // self.share_stride) * self.share_stride
        group_end = min(group_start + self.share_stride, self.n_layers)
        if self.kv_compute_position == 'first':
            return layer_idx == group_start
        else:
            return layer_idx == group_end - 1

    def forward(self, x, return_kv=False):
        B, N, C = x.shape
        cached_kv = {}

        for i in range(self.n_layers):
            h = self.lns[2 * i](x)
            q = self.q_projs[i](h).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

            group_idx = self._get_kv_group_idx(i)
            if self._should_compute_kv(i):
                kv = self.kv_projs[group_idx](h)
                k, v = kv.chunk(2, dim=-1)
                k = k.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
                v = v.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
                cached_kv[group_idx] = (k, v)
            else:
                k, v = cached_kv[group_idx]

            attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
            attn = attn.softmax(dim=-1)
            out = (attn @ v).transpose(1, 2).reshape(B, N, C)
            out = self.out_projs[i](out)
            x = x + out
            x = x + self.ffns[i](self.lns[2 * i + 1](x))

        if return_kv:
            return x, cached_kv
        return x


def measure_layer_similarity(model, x):
    """测量模型各层KV的余弦相似度，用于确定share_stride"""
    B, N, C = x.shape
    layer_kvs = {}

    h = x.clone()
    for i in range(model.n_layers):
        normed = model.lns[2 * i](h)
        group_idx = model._get_kv_group_idx(i)
        kv = model.kv_projs[group_idx](normed)
        k, v = kv.chunk(2, dim=-1)
        k = k.view(B, N, model.n_heads, model.head_dim)
        v = v.view(B, N, model.n_heads, model.head_dim)
        layer_kvs[i] = (k.detach().flatten(), v.detach().flatten())

        q = model.q_projs[i](normed).view(B, N, model.n_heads, model.head_dim).transpose(1, 2)
        k_t = k.transpose(1, 2)
        v_t = v.transpose(1, 2)
        attn = (q @ k_t.transpose(-2, -1)) * (model.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v_t).transpose(1, 2).reshape(B, N, C)
        out = model.out_projs[i](out)
        h = h + out
        h = h + model.ffns[i](model.lns[2 * i + 1](h))

    n = model.n_layers
    sim_matrix = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            ki, vi = layer_kvs[i]
            kj, vj = layer_kvs[j]
            k_sim = F.cosine_similarity(ki.unsqueeze(0), kj.unsqueeze(0)).item()
            v_sim = F.cosine_similarity(vi.unsqueeze(0), vj.unsqueeze(0)).item()
            sim_matrix[i, j] = (k_sim + v_sim) / 2

    return sim_matrix


n_layers = 6
model_shared = CrossLayerKVSharedTransformer(
    n_layers=n_layers, dim=64, n_heads=4, share_stride=2, kv_compute_position='first'
)

x = torch.randn(2, 16, 64)
out = model_shared(x)

print("=== 跨层KV共享 ===")
print(f"层数: {n_layers}")
print(f"共享步长s: 2 (每2层共享1组KV投影)")
print(f"KV计算位置: 组内第一层 (CLA做法)")
print(f"标准Transformer KV投影组数: {n_layers}")
print(f"共享后KV投影组数: {len(model_shared.kv_projs)}")

seq_len = 512
kv_per_layer = 1 * 4 * seq_len * 64 * 2 * 2
standard_kv_mem = n_layers * kv_per_layer / 1024 / 1024
shared_kv_mem = len(model_shared.kv_projs) * kv_per_layer / 1024 / 1024
print(f"\nKV Cache内存 (seq_len={seq_len}):")
print(f"  标准Transformer: {standard_kv_mem:.3f} MB")
print(f"  跨层共享(s=2): {shared_kv_mem:.3f} MB")
print(f"  节省: {(1 - shared_kv_mem / standard_kv_mem) * 100:.0f}%")
print(f"\n模型输出形状: {out.shape}")

print(f"\n=== 层间KV相似度测量（确定stride的依据）===")
sim_matrix = measure_layer_similarity(model_shared, x)
print(f"\n层间KV余弦相似度矩阵:")
header = "      " + "  ".join([f"L{j}" for j in range(n_layers)])
print(header)
for i in range(n_layers):
    row = f"L{i}   " + "  ".join([f"{sim_matrix[i, j]:.2f}" for j in range(n_layers)])
    print(row)

print(f"\n相邻层平均相似度: {torch.diag(sim_matrix, 1).mean().item():.4f}")
print(f"间隔1层平均相似度: {torch.diag(sim_matrix, 2).mean().item():.4f}")
print(f"\n结论: 相邻层相似度最高，s=2最安全；间隔越远相似度越低，s>2需谨慎")

print(f"\n=== 不同stride的KV投影组数对比 ===")
for s in [1, 2, 3, 4]:
    n_kv = (n_layers + s - 1) // s
    saving = (1 - n_kv / n_layers) * 100
    print(f"  s={s}: KV投影组数={n_kv}/{n_layers}, 节省={saving:.0f}%")

### KV Cache优化方法综合对比

不同KV Cache优化方法在内存节省、精度影响和适用场景方面的对比。实际部署中通常组合使用多种方法。

In [ ]:
print(f"{'方法':<25} {'内存节省':<15} {'精度影响':<15} {'适用场景':<25}")
print("-" * 80)
methods = [
    ("KV INT8量化", "50%", "极小", "通用"),
    ("KV INT4量化", "75%", "较小", "长序列/内存紧张"),
    ("PagedAttention", "消除碎片", "无", "多请求并发"),
    ("Prefix Caching", "共享前缀100%", "无", "多用户共享prompt"),
    ("滑动窗口", "固定O(W)", "丢失远距依赖", "短上下文/流式"),
    ("H2O淘汰", "可控", "较小", "长序列/预算受限"),
    ("跨层KV共享", "50%", "需验证", "层间相似度高"),
]
for name, saving, impact, scene in methods:
    print(f"{name:<25} {saving:<15} {impact:<15} {scene:<25}")

print(f"\n=== 产业级组合建议 ===")
print(f"1. 量化 + PagedAttention: 最通用的组合，vLLM默认方案")
print(f"2. 量化 + 滑动窗口: 端侧设备首选，内存可控")
print(f"3. Prefix Caching + 量化: 对话服务场景，复用system prompt")
print(f"4. H2O + 量化: 长文本生成，在有限预算内保留最重要KV")